In [ ]:
from opentrons import protocol_api
from time import *
from opentrons.types import *
import opentrons.execute
import json
import math
protocol = opentrons.execute.get_protocol_api("2.19")
protocol.home()

In [ ]:
# DECK SETUP AND LAYOUT

left_tiprack = protocol.load_labware("opentrons_96_tiprack_300ul", "6")
sample_plate = protocol.load_labware("corning_96_wellplate_360ul_flat", "3")
right_tiprack = protocol.load_labware("opentrons_96_tiprack_1000uL", "2")

# IMPORTING CUSTOM LABWARE
# Top plate holder. This is the rack that holds all top plates

with open('caber_topplt_holder_flush.json') as top_plate_file:
    top_plate_labware_def = json.load(top_plate_file)
    top_plate_rack = protocol.load_labware_from_definition(top_plate_labware_def, 9)
    
# Bottom plate. NOTE: JSON file is defined as a reservoir with one well. A1 position is at top left corner, a few mm offset. 
# This plate contains individual CaBER Plate measurement locations

with open('caber_bottom_plate.json') as bottom_plate_file:
    bottom_plate_labware_def = json.load(bottom_plate_file)
    bottom_plate = protocol.load_labware_from_definition(bottom_plate_labware_def, 10)


# DEFINING PIPETTES

left = protocol.load_instrument("p300_single_gen2", mount="left")
left.default_speed = 300

right = protocol.load_instrument("p1000_single_gen2", mount="right")

In [ ]:
# Making a class defining the individual CaBER plate measurement locations called CaberPlate
# The bottom_plate used below is the whole labware which is configured as a one-well reservoir containing multiple CaBER measurement locations.

class CaberPlate:
    bottom_plate_zero = bottom_plate["A1"].top(10)  # Top left corner of this bottom_plate labware. Considered our "zero location". 
                                                    # All CaBER plate measurement locations defined below are relative to this location.
    bottom_plate_base_z = -7.14     # The z value offset used in the methods below to define the location at which both top plate picked up by OT-2 pipette and CaBER plate on this labware touch.
    
    drop_dispense_z = -2            # The z value offset used in the methods below that prevents the pipette tip from crashing into a CaBER plate on this labware. 
                                    # This is the ideal height from which to dispense onto a CaBER Plate
    
    def __init__(self, name, diameter, x, y):
        self.name = name    # e.g. "sixmm"
        self.d = diameter   # diameter of an individual plate

        # x and y are used to define the measurement location of a given CaBER plate within the bottom_plate labware
        self.x = x          # x offset relative to bottom_plate_zero. This is the center location of a given plate location
        self.y = y          # y offset relative to bottom_plate_zero.
      
    
    def get_xy (self) -> Location: 
        """ 
        Returns the center location of the CaBER position relative to bottom_plate_zero. 
        Applies only x and y offsets relative to bottom_plate_zero while maintaining its 
        existing z-coordinate which is several mm above the true CaBER plate measurement location.
        """
        return CaberPlate.bottom_plate_zero.move(Point(self.x,self.y))
    

    def get_pltloc(self) -> Location: 
        """ 
        Returns the location at which both top plate held by the OT-2 pipette is centered over and contacts the bottom CaBER plate. 
        I.e. the point at which both top and bottom CaBER plates touch
        """
        return self.get_xy().move(Point(z = CaberPlate.bottom_plate_base_z))
    
    def get_droploc(self): 
        """
        Returns the location from which the sample droplent is dispensed onto the CaBER plate position.
        """
        return self.get_xy().move(Point(y = 0, z = CaberPlate.drop_dispense_z))
    
    def get_loadh(self): 
        """
        Returns the default separation distance between the CaBER top and bottom plates at the start of the experiment before strain is applied.
        Default aspect ratio is defined as h/r = 1, where h is the starting separation distance height, and r is the radius of the plate.
        Therefore, h is equal to half the plate diameter.
        
        E.g. a 6 mm plate would have a starting separation distance of 3 mm.
        """
        return self.d/2
    
    def get_loadvol(self): 
        """
        Returns the estimated default sample volume required to perfectly fill the starting gap between the CaBER top and bottom plates. 

        The volume is approximated as the volume of a cylinder with a diameter
        equal to the CaBER plate diameter and a height equal to the default starting separation distance
        """
        return math.pi*((self.d/2)**2)*self.get_loadh()

    
# These are individual CaBER plate default locations where a sample can be tested. Each plate location has a separate diameter
sixmm = CaberPlate("6mm plate", 6, 20.45, -31.15)
fourmm = CaberPlate("4mm plate", 4, 20.45,-41.2)
eightmm = CaberPlate("8mm plate", 8, 20.45,-51)

In [ ]:
# CABER PLATE LOCATION CALIBRATIONS
# From time to time, the OT-2 drifts. So, at the start of every day of experiments, or with a new set of top plates, 
# it is important to perform a calibration to ensure the top and CaBER bottom plates are properly aligned. 

def calibrate_caber_position(plt, pipette = left, x_offset=0, y_offset=0, z_offset=0):
    pipette.move_to(plt.get_xy())
    pipette.move_to(plt.get_pltloc().move(Point(z = z_offset, 
                                                x = x_offset, 
                                                y = y_offset)), speed = 5)

    confirm = input("Are the top and bottom plates aligned, and just begin to touch? enter y if true")
    if confirm.lower() == "y":
        print(f"Calibration complete. saved offsets: x_offset = {x_offset}, y_offset = {y_offset}, z_offset = {z_offset}")
        return(plt, x_offset, y_offset, z_offset)
    else:
        new_x_offset = input(f"New x_offset. Current = {x_offset}")
        new_y_offset = input(f"New y_offset. Current = {y_offset}")
        new_z_offset = input(f"New z_offset. Current = {z_offset}")
        return calibrate_caber_position(pipette = pipette, 
                                        plt = plt, 
                                        x_offset = float(new_x_offset), 
                                        y_offset = float(new_y_offset), 
                                        z_offset = float(new_z_offset))

In [ ]:
# The workflow below guides through calibrations

# Pick up tip! otherwise it WILL crash.
top_plate_rack.reset()
left.pick_up_tip(top_plate_rack)

# Calibrate
plt, x_offset, y_offset, z_offset = calibrate_caber_position(plt = sixmm, 
                                                        pipette = left, 
                                                        x_offset=0, 
                                                        y_offset=0, 
                                                        z_offset=0)

# Update CaberPlate locations with new offsets:
print("updating CaberPlate locations with new offsets")
plt.x += x_offset
plt.y += y_offset 
CaberPlate.bottom_plate_base_z += z_offset

left.drop_tip()

In [ ]:
# with this new calibration, cycle through pickup and positioning of the top plates across three different top plates to ensure calibration worked well.
plt = sixmm # update as needed

left.pick_up_tip(top_plate_rack)
left.move_to(plt.get_xy())
left.move_to(plt.get_pltloc(), speed = 5)
protocol.delay(seconds = 30) # Change pause time as needed 
left.drop_tip()

<InstrumentContext: p300_single_v2.1 in LEFT>

In [ ]:
# FUNCTIONS FOR HIGHSPEED CAMERA TRIGGERING 
import serial
import time

# Set up serial connection
# Replace the port name below with your actual Arduino port
PORT = '/dev/ttyACM0' # Change if OT-2 changes port. Easy find with bash command: dmesg | tail -n 20
BAUD = 115200

# Open the serial port
ser = serial.Serial(PORT, BAUD, timeout=1) -0.15 -0.38 -0.3
time.sleep(2)  # Give Arduino a moment to reset after opening connection

print("Connected to Arduino on", PORT)

def arm_camera(pulse_count=1, delay_between=1):
    """
    Sends 'pulse_count' trigger pulses to the Arduino.
    Each pulse makes pin D7 go HIGH for 10 ms.
    delay_between = delay in seconds between pulses.
    Used to make Photron go from "Record" to "Ready". 
    For automation workflows, ensure that autosave is enabled in the fastcam software 
    and I/O port "Input 1" is configured to arm/ready.
    """
    for i in range(pulse_count):
        ser.write(b'r')
        print(f"Pulse {i+1} sent")
        time.sleep(delay_between)
        
        
def trigger_camera(pulse_count=1, delay_between=1):
    """
    Sends 'pulse_count' trigger pulses to the Arduino.
    Each pulse makes pin D8 go HIGH for 10 ms.
    delay_between = delay in seconds between pulses.
    Used to make Photron go from "Ready" to recording to memory.
    For automation workflows, ensure that autosave is enabled in the fastcam software 
    and I/O port "Input 2" is configured to record.
    """
    for i in range(pulse_count):
        ser.write(b't')              # send the single-byte trigger command
        print(f"Pulse {i+1} sent")
        time.sleep(delay_between)    # wait before next pulse

In [ ]:
## AERIS OPERATION FUNCTIONS 

# Aspirate and dispensing of complex solutions:
def viscous_aspirate(loadv, vincr, sample_loc, tdelay, final_wait): 
    """ 
    Aspirates viscous solutions: uses a 1mL pipette intact tip, roughly works upto 2%HA
    This method breaks up the aspiration of sample into multiple increments, pausing between each increment
    loadv = total volume to aspirate (load volume), float
    vincr = increment volume to use when aspirating,  float
    sample_loc = location on well plate from which to take the sample
    tdelay =  time, in seconds,  between each aspiration increment
    final_wait = the time, in seconds, to wait after the final increment"""
    v = 0 # Volume aspirated 
    while v < loadv:
        if loadv - v > vincr:
            right.aspirate(vincr, location = sample_loc)
            protocol.delay(seconds = tdelay)
            v += vincr 
        else: 
            right.aspirate(loadv-v, location = sample_loc)
            protocol.delay(seconds = tdelay)
            v += loadv-v
    protocol.delay(seconds = final_wait)
    right.touch_tip(radius = 0.8, v_offset = -0.15, speed = 5)  # touch sides of tip to well to minimize sample transfer

def viscous_dispense(dispv, vincr, plt, tdelay, final_wait, drop_h: float = 0): # 1mL pipette intact tip, roughly works upto 2%HA
    """
    Dispenses viscous solutions on specified caber plate:
    uses a 1mL pipette with intact tip, roughly works upto 2%HA
    dispv = total dispense volume, float
    vincr = increment volume to dispense, float
    plt = the plate location to dispense at. Specify CaberPlate object e.g.sixmm, fourmm etc.
    tdelay = time, in seconds between each dispense increments = float
    final_wait = the time, in seconds, after the final step = float
    drop_h = height from which to dispense =  Float > 0. Default is 0. 
    """ 
    v = 0 # Volume dispensed
    while v < dispv:
        if dispv - v > vincr:
            right.dispense(vincr, location = plt.get_droploc().move(Point(z = drop_h)))
            protocol.delay(seconds = tdelay)
            v += vincr
        else: 
            right.dispense(dispv-v, location = plt.get_droploc().move(Point(z = drop_h)))
            protocol.delay(seconds = tdelay)
            v += dispv-v
    protocol.delay(seconds = final_wait)



def pick_up_gel(plt, sample_loc, gel_pickup_wait: int):
    """Picks up gel from the sample plate
    Parameters:
        plt: CaberPlate
            The testing plate
        sample_loc: opentrons.protocol_api.labware.Well
            Sample location of format: sample_plate.wells()[0]
        gel_pick_up_wait: int 
            Time to wait after making contact with the gel, seconds"""
    left.pick_up_tip(top_plate_rack)
    left.move_to(sample_loc.top(z = 15))
    left.move_to(sample_loc.top(z = 1))
    left.move_to(sample_loc.bottom(z = 0), speed = 3)
    protocol.delay(seconds = gel_pickup_wait)
    left.move_to(sample_loc.top(z = 30), speed = 3)    
    

# SAMPLE LOADING
def caber_load_sample(plt, sample_loc, avincr, tdelay, dvincr, final_wait, vol_offset: float = -10, drop_h: float = 0): 
    """
    Aspirates sample from designated well location, and dispenses at designated CaberPlate location. 
    Note aspirate/dispense speed is 716ul/s. Change globally if needed for more viscous samples.
    Load and dispense volumes are calculated based on the CaberPlate object, starting height, and a volume offset
    
    plt = CaberPlate object e.g. sixmm, fourmm etc.
    sample_loc = location on well plate from which to take the sample
    vol_offset = volume offset, in uL, to define how much volume you actually want to load.
    avincr = increment volume to use when aspirating, float
    dvincr = increment volume to use when dispensing, float
    tdelay = time, in seconds between each aspirate or dispense increment = float
    final_wait = the time, in seconds, after the final increment of sample is aspirated or dispensed= float
    drop_h = height from which to dispense sample onto the Caber plate.
    """ 
     
    loadv = (plt.get_loadvol()+vol_offset)+((plt.get_loadvol()+vol_offset)*0.2) # Calculating load volume
    dispv = plt.get_loadvol()+vol_offset # Calculating dispense volume
    
    right.pick_up_tip(right_tiprack)
    viscous_aspirate(loadv, avincr, sample_loc, tdelay, final_wait)  # Aspirate sample
    
    viscous_dispense(dispv, dvincr, plt, tdelay, final_wait, drop_h) # Dispense sample
    right.move_to(plt.get_xy())
    right.drop_tip()


# SAMPLE ENTRAPMENT
def caber_sandwich(plt, dipf, sp, tsoak, lh_offset: float = 0, plt_pause_time: float = 0):
    """ 
    Picks up the top plate from the top plate rack, and entraps the sample between the two plates
    plt = CaberPlate object. e.g sixmm, fourmm, etc.
    dipf = dip factor, how far into the sample the top plate should descend to make full contact, 
        before lifting back up to the load height
    sp = speed of plate dipping movement in mm/s
    lh_offset = how much lower you want the plate to go from preset (height/radius of plate = 1). Positive = higher, negative = lower, in mm 
        Do not change unless absolutely necessary. 
    tsoak = resting time to enable sample relaxation before plats are pulled apart
    plt_pause_time = time to pause after picking up the plate"""
    
    left.pick_up_tip(top_plate_rack)        # Pick up tip.
    protocol.delay(seconds = plt_pause_time)
    
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh())))
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh()-(plt.get_loadh()*dipf))), speed = sp) # move into sample to make full contact
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh()+lh_offset)), speed = sp)  # Move to final height
    protocol.delay(seconds = tsoak) # let sample soak

def caber_load_gel(plt, dipf, sp, tsoak, lh_offset: float = 0):
    """ 
    Picks up the top plate from the top plate rack, and entraps the sample between the two plates
    plt = CaberPlate object. e.g sixmm, fourmm, etc.
    dipf = dip factor, how far into the sample the top plate should descend to make full contact, 
        before lifting back up to the load height
    sp = speed of plate dipping movement in mm/s
    lh_offset = how much lower you want the plate to go from preset (height/radius of plate = 1). Positive = higher, negative = lower, in mm 
        Do not change unless absolutely necessary. 
    tsoak = resting time to enable sample relaxation before plats are pulled apart
    plt_pause_time = time to pause after picking up the plate"""
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh())))
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh()-(plt.get_loadh()*dipf))), speed = sp) # move into sample to make full contact
    left.move_to(plt.get_pltloc().move(Point(z = plt.get_loadh()+lh_offset)), speed = sp)  # Move to final height
    protocol.delay(seconds = tsoak)
    
# TESTING

# defining rate at which to lift
def plt_lift_rate (h: float, actuation_time: float) -> float: 
    """Defines the rate at which to lift the top plate away from sample. Returns rate as float in mm/s
    h = height from start height to which to lift in mm. It is effectively the delta h: float
    actuation_time = how quickly should lift in ms"""
    lift_rate = h/actuation_time*1000
    return lift_rate

def caber_test_sample(plt, h, actuation_time):
    """
    Lifts top plate up from sample to a defined height and at a defined speed. 
    plt = CaberPlate object e.g. sixmm
    h = height from the point the plates touch (so from base of palte) to which to lift in mm. It is the total h: float
    actuation_time = how quickly should lift in ms"""
    left.move_to(plt.get_pltloc().move(Point(z = h)), speed = plt_lift_rate(h,actuation_time)) # lift plate


# final function that combines it all
def caber_run_iter_visc_autocam (plt, sol, trig_wait_t, avincr, tdelay, dvincr, final_wait, dipf, sp, tsoak, h, actuation_time, lh_offset: float = 0, plt_pause_time: float = 0, drop_h: float = 0, vol_offset: float = -10):
    """
    Orchestrates the entire workflow of loading a sample, entraping it between the plates, and lifting the top plate while triggering the camera. This workflow requires manual cleaning.
    sol = list of locations where solutions to be tested are stored.
    trig_wait_t = time to wait after sample is loaded and plates are entraped before triggering camera"""
    print(sol)
    cam_ready = input("Change 'trigger mode' to 'end' on Photron UX mini. Hit record to arm camera. Type y when done") # check to see if 
    if cam_ready.lower() == "y":    
        for sample_loc in sol:
            clean_stage = input("Clean the stage. Type y when done")
            if clean_stage.lower() == "y":
                print(sample_loc)
                caber_load_sample(plt, sample_loc, avincr, tdelay, dvincr, final_wait, vol_offset=vol_offset, drop_h=drop_h)
                caber_sandwich(plt, dipf, sp, tsoak, lh_offset, plt_pause_time)
                caber_test_sample(plt, h, actuation_time)
                protocol.delay(seconds = trig_wait_t)
                trigger_camera()
                left.drop_tip()
    else:
        print("Set up the camera and hit run again.")

In [ ]:
## OPERATING PARAMETERS - DEFAULTS - Do not change. Any deviations from these defaults should be made in the caber_run_iter_visc_autocam function call below.

# Right pipette = sample transfer
# Left pipette = CaBER

# Setting arm speeds: 
left.default_speed = 400 #NOTE going to 100 gets OT2 to get mad
right.default_speed = 400

#Flow rates and liquid handling parameters:
right.flow_rate.aspirate = 2.5 
right.flow_rate.dispense = 5 

# NOTE: Need atleast one increment for the code to work, do not set to zero
avincr = 10 # volume increments to aspirate = uL
dvincr = 10 # volume increments to dispense = uL
tdelay = 30 # time to pause between aspirate/dispense increments = seconds
final_wait = 60 # time to wait after final aspirate/dispense = seconds  
vol_offset = -10 #how much more or less (volume, uL) you need to get the perfect sample loading. + is more, - is less


#plate soaking parameters:
lh_offset = 0 #how much lower you want the plate to go from preset (h/r=1). positive = higher, negative = lower. 
                #Do not change unless absolutely necessary
dipf = 0.2 # dipf is dip factor, how far into the sample do you want the plate to go to make full contact 
            # before receding to the final height
sp = 0.5  #speed in mm/s  is how slowly we want the plate to go into and out of the sample to make full contact
tsoak = 5 #soak time in seconds

# plate lifting parameters:
h = 10 # height to which to lift in mm
actuation_time = 45 #how quickly we want to lift in ms

In [ ]:
# RUNNING THE EXPERIMENT - PIPETTABLE SOLUTIONS

### Resetting pipettes in case they have something attached ###
# right.drop_tip()
# left.drop_tip()

### Resetting tipracks ###
top_plate_rack.reset()
# right_tiprack.reset()

### Setting flow rates and interval pipetting parameters for viscous liquid handling ###
right.flow_rate.aspirate = 100
right.flow_rate.dispense = 100
avincr = 10
tdelay = 5
dvincr = 10
final_wait = 10

### Setting plate lifting parameters: ###
h = 8 # height to which to lift in mm (final height from base of plate)
actuation_time = 30 #how quickly we want to lift in ms
dipf = 0.3
lh_offset = 0

### Setting camera triggering parameters: ###
trig_wait_t= 2

reps = 1 # set number of replicates. If loaded reps in different wells, set reps = 1. If need to return to the wells for replicates, increase number of reps accordingly.
for r in range(reps):
    caber_run_iter_visc_autocam (plt = sixmm, 
                                 sol = sample_plate.wells()[9:10],
                                 trig_wait_t = trig_wait_t, 
                                 avincr = avincr, 
                                 tdelay = tdelay, 
                                 dvincr = dvincr, 
                                 final_wait = final_wait, 
                                 dipf = dipf, 
                                 sp = sp, 
                                 tsoak = tsoak, 
                                 h = h,
                                 actuation_time = actuation_time,
                                 vol_offset = 0,
                                 drop_h = 2,
                                 lh_offset = 0)
    print(f"completed rep#{r+1}")
print(f"all {reps} runs complete")

In [ ]:
# RUNNING THE EXPERIMENT - HYDROGELS

### Resetting pipettes in case they have something attached ###
# right.drop_tip()
# left.drop_tip()

### Resetting tipracks ###
top_plate_rack.reset()
# right_tiprack.reset()

### Setting plate lifting parameters: ###
h = 8 # height to which to lift in mm (final height from base of plate)
actuation_time = 30 #how quickly we want to lift in ms


### Setting camera triggering parameters: ###
trig_wait_t = 30 # seconds

### Setting plate and sample locations: ###
plt = sixmm 
sol = sample_plate.wells()[9:10]

# Run
cam_ready = input("Change 'trigger mode' to 'end' on Photron UX mini. Hit record to arm camera. Type y when done") # check to see if 
if cam_ready.lower() == "y":  
    for s in sol:
        clean_stage = input("Clean the stage. Type y when done")
        if clean_stage.lower() == "y":
            # Pick up gel
            pick_up_gel(plt = plt,
                    sample_loc = s,
                        gel_pickup_wait = 15)
            # Sandwich
            caber_load_gel(plt = plt, 
                            dipf = 0.2, 
                            sp = 0.5, 
                            tsoak = 60, 
                            lh_offset = 0)
                
            # test
            caber_test_sample(plt = plt, 
                                h = h, 
                                actuation_time = 15)
            
            protocol.delay(seconds = trig_wait_t)
            trigger_camera()
            left.drop_tip()
                
